# Gate 1 - Source identity and access

Checklist: record the authoritative source and confirm its size, row count, column count and header. On any mismatch, record an anomaly; never substitute another file.

**Manual items (not automated):** load project materials into the team Colab space; confirm fellows, coach and Challenge Advisor have access.

In [1]:
import csv, hashlib, io
from pathlib import Path

def find_root():
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / 'data' / 'allstate_claims_data.csv').exists():
            return d
    raise FileNotFoundError('data/allstate_claims_data.csv not found')

SOURCE = find_root() / 'data' / 'allstate_claims_data.csv'   # authoritative source
EXPECTED = {'size_bytes': 70_025_339, 'rows': 188_318, 'columns': 132}
EXPECTED_HEADER = ['id'] + [f'cat{i}' for i in range(1,117)] + [f'cont{i}' for i in range(1,15)] + ['loss']
print('Authoritative source: data/allstate_claims_data.csv')

Authoritative source: data/allstate_claims_data.csv


## Source identity
The size is checked on the raw bytes on disk and again with CRLF normalized to LF (the form Git stores).

In [2]:
raw = SOURCE.read_bytes()
lf = raw.replace(b'\r\n', b'\n')
with SOURCE.open(newline='') as f:
    r = csv.reader(f); header = next(r); rows = sum(1 for _ in r)

obs = {'size_bytes_on_disk': len(raw), 'size_bytes_lf_normalized': len(lf),
       'crlf_count': raw.count(b'\r\n'), 'rows': rows, 'columns': len(header)}
obs['sha256_lf_normalized'] = hashlib.sha256(lf).hexdigest()
obs['sha256_on_disk'] = hashlib.sha256(raw).hexdigest()
for k, v in obs.items(): print(f'{k:28} {v}')

size_bytes_on_disk           70213658
size_bytes_lf_normalized     70025339
crlf_count                   188319
rows                         188318
columns                      132
sha256_lf_normalized         74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03
sha256_on_disk               9b70f35b548ae294cfba10e549a2c777c11c54c66ae452ea54eed9f4148fd2f5


In [3]:
checks = {
 'size on disk == expected':        obs['size_bytes_on_disk'] == EXPECTED['size_bytes'],
 'size LF-normalized == expected':  obs['size_bytes_lf_normalized'] == EXPECTED['size_bytes'],
 'data rows == 188,318':            obs['rows'] == EXPECTED['rows'],
 'columns == 132':                  obs['columns'] == EXPECTED['columns'],
 'header exact and in order':       header == EXPECTED_HEADER,
}
for k, v in checks.items(): print('PASS' if v else 'FAIL', '-', k)

FAIL - size on disk == expected
PASS - size LF-normalized == expected
PASS - data rows == 188,318
PASS - columns == 132
PASS - header exact and in order


## Anomaly register entry

In [4]:
anomalies = []
if not checks['size on disk == expected']:
    anomalies.append({
      'id': 'A-001', 'title': 'On-disk file size differs from expected (CRLF line endings)',
      'severity': 'non-blocking' if checks['size LF-normalized == expected'] else 'BLOCKING',
      'owner': 'Junaid Pathan', 'status': 'open',
      'evidence': f"on disk {obs['size_bytes_on_disk']:,} B; {obs['crlf_count']:,} CRLF; LF-normalized {obs['size_bytes_lf_normalized']:,} B; expected {EXPECTED['size_bytes']:,} B",
      'handling': 'Windows checkout converts LF to CRLF (Git stores LF). Content identical; verify size/hash on LF-normalized bytes. Do not alter the file.',
      'impact': 'None on data values; would only fail a naive byte-size or hash check on Windows.'})
for a in anomalies: print(a)
assert all(checks[k] for k in ['data rows == 188,318','columns == 132','header exact and in order','size LF-normalized == expected']), 'BLOCKING mismatch - stop'
print('\nSource identity confirmed (size verified on LF-normalized bytes).')

{'id': 'A-001', 'title': 'On-disk file size differs from expected (CRLF line endings)', 'severity': 'non-blocking', 'owner': 'Junaid Pathan', 'status': 'open', 'evidence': 'on disk 70,213,658 B; 188,319 CRLF; LF-normalized 70,025,339 B; expected 70,025,339 B', 'handling': 'Windows checkout converts LF to CRLF (Git stores LF). Content identical; verify size/hash on LF-normalized bytes. Do not alter the file.', 'impact': 'None on data values; would only fail a naive byte-size or hash check on Windows.'}

Source identity confirmed (size verified on LF-normalized bytes).
